In [1]:
# 1. 크롬 드라이버 셀레니움 환경 설정하기
## 크롬드라이버 설치 -> 크롬 정보 확인하여 버전에 맞는걸로 Download
### https://googlechromelabs.github.io/chrome-for-testing/#stable

# 2. 셀레니움 설치 완료 후 필요한 모듈 불러오기
from selenium import webdriver                                          # webdriver : 크롬 가상 드라이버 실행
from selenium.webdriver.common.by import By                             # By : 실제 페이지 내용을 긁어올 때 수단을 설정하는 모듈
from selenium.webdriver.common.keys import Keys                         # Keys : 대신 키보드를 눌러주는 동작을 전달
from selenium.webdriver.support.ui import WebDriverWait                 # WebDriverWait : 대기가 필요한 상황에서 사용하기
from selenium.webdriver.support import expected_conditions as EC        # expected_conditions : 요소의 상태를 알아볼 때 사용
from selenium.webdriver.chrome.options import Options                   # Options : 크롬 드라이버의 옵션을 정함. ex) 크롬창 크기 등
from selenium.webdriver.chrome.service import Service                   # Service : Selenium에서 ChromeDriver의 경로를 설정하고 관리하는 역할
from selenium.webdriver.common.action_chains import ActionChains        # ActionCahins : Selenium에서 입력 장치를 사용하여 복잡한 사용자 동작을 자동화
from selenium.common.exceptions import NoSuchElementException           # NoSuchElementException : 지정한 조건에 맞는 요소를 찾을 수 없을 때 발생하는 예외 함수

# 3. BeautifulSoup 활용 할 모듈 불러오기
# from bs4 import BeautifulSoup                       
import time                                     # 시간 관련 작업을 간편하게 처리하기 위해 사용
# import math                                   # 수학적 계산을 위한 다양한 함수들을 제공
import pandas as pd 
# import pickle                                 # 데이터를 저장할 때 사용

# 4. 실행 편의를 위한 라이브러리 
from tqdm import tqdm                                   # 진행률 보여주는 라이브러리 
import datetime                                         
import ipywidgets as widgets                            # 입력받을 위젯 사용 라이브러리 
from IPython.display import display, clear_output
import random

### 장르 선택 관련 함수 

In [2]:
# 이동할 장르의 xpath를 설정하는 함수  [{selected_genre}, {selected_value[index]}]
def make_xpath (genre_dict) :
    print(f"make_xpath 함수 호출 \n {genre_dict}")
    gnr_code = "GN0"+genre_dict["menu_key"]
    span_text = genre_dict["genre"]
    xpath = f"//li/a[contains(@href, 'gnrCode={gnr_code}')]/span[text()='{span_text}']"

    return xpath 

In [3]:
# 장르 클릭하는 함수 이렇게 밖에 못하나... 
def click_genre (genre_dict, driver) : 
    # xpath 가져오기 
    xpath = "" 
    if genre_dict : 
        xpath = make_xpath (genre_dict)

    # 장르 선택하여 클릭
    if xpath != "" :
        second_box = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.XPATH,xpath)))
        second_box.click()

In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 장르 메뉴 정의
menu_abroad = {
    "록/메탈": "1000",
    "R&B/Soul": "1300"
}

menu_korea = {
    "발라드": "100",
    "랩/힙합": "300",
    "R&B/Soul": "400",
    "인디음악": "500",
    "트로트": "600",
}

def genre_selection(driver):
    # 반환할 딕셔너리 초기화
    ret_dict = {}

    # 국가 선택 위젯
    country_selector = widgets.Dropdown(
        options=[('국가를 선택해주십시오.', '0'), ('국내', '1'), ('국외', '2')],
        description='국가 선택:',
    )

    # 메뉴 선택 위젯, 초기에는 비어있음
    menu_selector = widgets.Dropdown(
        options=[],
        description='메뉴 선택:',
    )

    # 메뉴 출력 위젯
    menu_output = widgets.Output()

    def show_menu(change):
        """국가 선택 시 해당 메뉴를 보여주는 함수"""
        with menu_output:
            clear_output()  # 이전 출력을 지움
            if change['new'] == '1':
                menu = menu_korea
            elif change['new'] == '2':
                menu = menu_abroad
            else:
                menu = {}
                menu_selector.options = []
                return
            
            # 메뉴 옵션 업데이트
            menu_selector.options = list(menu.keys())
            print("메뉴가 업데이트되었습니다. 선택하세요.")
            menu_selector.layout.display = 'block'  # 메뉴 선택 레이아웃 표시

    # 메뉴 선택 후 해당 장르를 반환하는 함수
    def select_genre(change):
        nonlocal ret_dict  # 외부 변수에 접근
        selected_genre = change['new']
        if selected_genre:
            selected_value = list(menu_korea.values()) if country_selector.value == '1' else list(menu_abroad.values())
            index = menu_selector.options.index(selected_genre)
            ret_dict.update({
                "genre": selected_genre,
                "menu_key": selected_value[index]
            })
            #장르 클릭 
            click_genre(ret_dict,driver)
            with menu_output:
                clear_output()
                print(f"선택한 장르: {selected_genre}, 코드: {selected_value[index]}")
            # 메뉴 선택 후 위젯 숨기기
            menu_selector.layout.display = 'none'  # 메뉴 숨기기
            print("메뉴 선택 후 위젯 숨기기 후",ret_dict) 

    # 국가 선택 시 메뉴 보여주기
    country_selector.observe(show_menu, names='value')

    # 메뉴 선택 시 장르 출력
    menu_selector.observe(select_genre, names='value')

    # 위젯 표시
    display(country_selector, menu_output, menu_selector)

    # 장르 선택 결과가 나올 때까지 대기하는 대신, 결과를 반환
    return ret_dict

# 함수를 호출하여 실행
# result = genre_selection()
# print("선택된 장르 정보:", result)


### 가사 수집 관련 함수

In [5]:
def collect_lyrics_in_list () : 
    columns = ['title', 'artist', 'lyrics', 'likes']
    song_data = pd.DataFrame(columns=columns)

    # 멜론 차트 list 만들기
    title = []
    artist = []
    lyrics = []
    likes = []

    # 기본 설정 값
    table_all = driver.find_element(By.XPATH, '//*[@id="songList"]')
    list_all = driver.find_elements(By.CSS_SELECTOR, "tbody>tr")

    # 수집 루틴
    # tqdm 라이브러리로 진행 상황 바 표시
    for idx, meta in tqdm(enumerate(list_all), total=len(list_all), desc="Processing songs"):
        time.sleep(random.uniform(1,3)) #   1~3초 사이 랜덤 시간으로 쉼
        try :
            # 스크롤 동적으로 이동 : 인덱스 기반 스크롤
            driver.execute_script(f"window.scrollTo(0, {idx * 85});")

            # 특정 요소가 로드될 때까지 대기
            WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.CSS_SELECTOR, "tbody>tr")))
            
            # 위치 찾아서 진입
            table_all = driver.find_element(By.XPATH, '//*[@id="songList"]')
            list_all = driver.find_elements(By.CSS_SELECTOR, "tbody>tr")

            # 현재 곡 클릭
            list_click = list_all[idx].find_elements(By.CSS_SELECTOR, "td")
            list_click[3].click()
            song_click = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.CSS_SELECTOR, f"#frm > div > table > tbody > tr:nth-child({int(idx+1)}) > td:nth-child(4) > div > a:not(.section_hitsong)")))
            song_click.click()

            # 제목 추가
            title_text = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.CSS_SELECTOR, ".song_name"))).text
            title.append(title_text)

            # 아티스트 이름 추가
            artist_name = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.CSS_SELECTOR, ".artist_name"))).text
            artist.append(artist_name)
            
            # 가사 불러오기 
            # lyricArea 요소 찾기
            lyric_area = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.ID, "lyricArea")))

            # lyricArea 안의 div 요소 찾기
            lyric_div = lyric_area.find_element(By.CSS_SELECTOR, 'div')

            # 클래스 이름 판별
            if 'lyric' in lyric_div.get_attribute('class'):
                # print("클래스가 'lyric'입니다.")
                lyrics_text = lyric_div.text  
            elif 'lyric_none' in lyric_div.get_attribute('class'):
                # print("클래스가 'lyric_none'입니다.")
                lyrics_text = ""
            else:
                # print("클래스가 'lyric_none'도 아니고 'lyric'도 아닙니다.")
                lyrics_text = ""
            lyrics.append(lyrics_text)

            # 좋아요 개수 추가
            likes_text = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.CSS_SELECTOR, ".cnt"))).text
            likes.append(likes_text)

            # 뒤로가기
            driver.back()
            WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.XPATH, '//*[@id="songList"]')))
            
        except :
            print(f"오류 발생", {idx})        
            break
        
        # 곡 정보 저장
        song_data = pd.DataFrame(
            {
                "title": title,
                "artist": artist,
                "lyrics": lyrics,
                "likes": likes
            }
        )

    return song_data

In [6]:
import datetime

def make_to_csv (merge_songs) : 

    #데이터 프레임 저장
    address = '../01_data_모음/'

    # 현재 시간 가져오기
    now = datetime.datetime.now()
    # 시간 형식 지정 (예: '2025-01-15_14-30-00')
    timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")

    # 파일 이름 생성
    file_name = f"melon_{timestamp}.csv"

    # song_data.to_csv(address, index=False, encoding='utf-8-sig')
    merge_songs.to_csv(path_or_buf=address+file_name)

    return print(f"{file_name}이 저장되었습니다.")

### 소스 실행부 

In [1]:
# 크롤링 화면 진입 
service = Service(r"C:\Two_Kim_and_One_Lee\melon\chromedriver-win64\chromedriver.exe")  # ChromeDriver 경로 (지민) ** 각자 세팅 해주세요! 
options = webdriver.ChromeOptions()

# 브라우저 열기
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 10)

# 멜론뮤직 홈페이지로 이동
url = "https://www.melon.com/"
driver.maximize_window()

driver.get(url)                                             # 멜론 뮤직 차트로 진입
time.sleep(2)                                               # 페이지 로드 대기

# 멜론뮤직 '장르음악' 선택
melon_ganre = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.XPATH, '//*[@id="gnb_menu"]/ul[1]/li[3]/a/span[2]')))
melon_ganre.click()

# 장르 선택 함수를 호출
genre_selection(driver)

# '인기순' 클릭
popular_box = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.XPATH, '//*[@id="frm"]/div/div/div/a[2]')))
popular_box.click()

NameError: name 'Service' is not defined

In [ ]:
# 가사 수집 실행 
columns = ['title', 'artist', 'lyrics', 'likes']
merge_songs = pd.DataFrame(columns=columns)

idx = 1
while idx <= 1000:
    try:
        curr_songs = collect_lyrics_in_list()  # 곡 정보 가져오기
        merge_songs = pd.concat([merge_songs, curr_songs], ignore_index=True)

        # 10n + 1 번째 (11, 21, 31...) 에는 'NEXT' 버튼 클릭
        if idx > 1 and idx % 10 == 1:
            try:
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.next'))
                )
                next_button.click()
            except Exception as e:
                print("NEXT 버튼을 찾을 수 없음:", e)
                break  # 페이지 이동이 불가능하면 종료

        # 10n + 1이 아닐 때는 페이지 번호 버튼 클릭
        else:
            try:
                page_buttons = driver.find_elements(By.XPATH, '//*[@id="pageObjNavgation"]/div/span/a')
                if len(page_buttons) > 0:
                    page_buttons[min(len(page_buttons) - 1, (idx - 1) % 10)].click()
                else:
                    print("페이지 버튼을 찾을 수 없음")
                    break
            except Exception as e:
                print("페이지 버튼 클릭 실패:", e)
                break

        idx += 1  # 다음 페이지로 넘어가기 위해 idx 증가

    except Exception as e:
        print("다음 페이지가 없습니다:", e)
        make_to_csv(merge_songs)
        break  # 페이지 이동이 불가능하면 반복 종료


In [13]:
merge_songs

,title,artist,lyrics,likes
0,나는 반딧불,황가람,나는 내가 빛나는 별인 줄 알았어요\n한 번도 의심한 적 없었죠\n몰랐어요 난 내가...,"88,439"
1,내게 사랑이 뭐냐고 물어본다면,로이킴,뜨겁게 사랑했던\n계절을 지나\n처음과는 조금은 달라진\n우리 모습을\n걱정 하진 ...,"50,807"
2,소나기,이클립스 (ECLIPSE),그치지 않기를 바랬죠\n처음 그대 내게로 오던 그날에\n잠시 동안 적시는\n그런 비...,"170,100"
3,천상연,이창섭,아니길 바랬었어\n꿈이길 기도했지\n너 없는 가슴으로 살아가야 하는 건\n내게는 너...,"115,418"
4,슬픈 초대장,순순희(지환),내 야윈 손위로 온 초대장 위에\n널 데려간다는 그와 네 이름\n오래전 헤어지던 날...,"61,624"
...,...,...,...,...
2128,하루하루,김보경 (NEON),거짓말 날 위해 하는 말\n\n혼잣말 버릇이 된 이말\n\n괜찮아 질 거야 누구나\...,"20,833"
2129,12월 32일,서은광 (비투비),올해가 가기 전에 꼭 돌아온다고\n걱정하지 말고 기다리면 된다고\n기다렸던 만큼 우...,"7,068"
2130,좋은 날,멜로망스,조용한 바람\n그대 생각 하나\n내게 물어옵니다\n그렇게 그댄\n어느새 내 맘에\n...,"16,311"
2131,어쩌다 너를,Hanul,틀어진 맘을 그냥 버려두기엔\n아직 서투른 내사랑이 너무 가엾잖아\n이렇게 아픈사랑...,"8,309"


In [14]:
make_to_csv(merge_songs)

melon_2025-01-20_13-27-50.csv이 저장되었습니다.
